# utils

## utils.metrics

In [ ]:
import pyiqa
import torch.nn as nn
from torch import Tensor


class ImageQualityMetrics(nn.Module):
    def __init__(self, device: str = "cuda") -> None:
        super().__init__()
        self.device_type: str = device

        self.psnr: nn.Module = pyiqa.create_metric(
            metric_name="psnr",
            device=device,
        )
        self.ssim: nn.Module = pyiqa.create_metric(
            metric_name="ssim",
            device=device,
        )
        self.lpips: nn.Module = pyiqa.create_metric(
            metric_name="lpips",
            device=device,
        )
        self.brisque: nn.Module = pyiqa.create_metric(
            metric_name="brisque",
            device=device,
        )
        self.niqe: nn.Module = pyiqa.create_metric(
            metric_name="niqe",
            device=device,
        )

    def forward(self, preds: Tensor, targets: Tensor) -> dict[str, float]:
        preds = preds.to(device=self.device_type)
        targets = targets.to(device=self.device_type)

        return {
            "PSNR": self.psnr(preds, targets).mean().item(),
            "SSIM": self.ssim(preds, targets).mean().item(),
            "LPIPS": self.lpips(preds, targets).mean().item(),
        }

    def no_ref(self, preds: Tensor) -> dict[str, float]:
        preds = preds.to(device=self.device_type)

        return {
            "BRISQUE": self.brisque(preds).mean().item(),
            "NIQE": self.niqe(preds).mean().item(),
        }

    def full(self, preds: Tensor, targets: Tensor) -> dict[str, float]:
        ref_metrics = self.forward(preds=preds, targets=targets)
        no_ref_metrics = self.no_ref(preds=preds)
        return {**ref_metrics, **no_ref_metrics}

## utils.utils

In [ ]:
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import torch.nn as nn
import torchvision.transforms.functional as F
from torch import Tensor
from torchinfo import summary
from torchvision.utils import save_image


def show_batch(images: Tensor, ncols: int = 8) -> None:
    nimgs: int = images.shape[0]
    nrows: int = (nimgs + ncols - 1) // ncols
    plt.figure(figsize=(ncols * 3, nrows * 3))
    for i in range(nimgs):
        plt.subplot(nrows, ncols, i + 1)
        plt.imshow(X=F.to_pil_image(pic=images[i]))
        plt.axis("off")
        plt.title(label=f"Image {i}")
    plt.tight_layout()
    plt.show()


def make_dirs(path: str) -> str:
    path_obj: Path = Path(path)
    path_obj.mkdir(parents=True, exist_ok=True)
    return path


def print_metrics(metrics: dict[str, float], prefix: str = "") -> None:
    for key, value in metrics.items():
        print(f"{prefix}{key}: {value:.4f}")


def save_images(
    batch_list: list[list[Tensor]],
    out_dir: Path,
    prefix: str = "infer",
    ext: str = "png",
) -> None:
    for i, datasets in enumerate(iterable=batch_list):
        save_path_str: str = make_dirs(path=f"{out_dir}/batch{i + 1}")
        for ii, batch in enumerate(iterable=datasets):
            save_image(
                tensor=batch,
                fp=f"{save_path_str}/{prefix}_{ii:04d}.{ext}",
                nrow=8,
                padding=2,
                normalize=True,
                value_range=(0, 1),
            )


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def summarize_model(
    model: nn.Module,
    input_size: list[int] | list[list[int]] | None = None,
    input_data: Any = None,
    **kwargs: Any,
) -> Any:
    if input_data is not None:
        return summary(model=model, input_data=input_data, **kwargs)
    if input_size is not None:
        return summary(model=model, input_size=input_size, **kwargs)
    raise ValueError("Either input_data or input_size must be provided.")


def weights_init(m: nn.Module) -> None:
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.xavier_normal_(tensor=m.weight)
        if m.bias is not None:
            nn.init.constant_(tensor=m.bias, val=0.0)

    elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
        nn.init.constant_(tensor=m.weight, val=1.0)
        nn.init.constant_(tensor=m.bias, val=0.0)

# data

## data.utils

In [ ]:
import random

from pathlib import Path
from typing import Tuple, cast

from PIL import Image
from torch import Tensor
from torch.utils.data import Dataset
from torchvision import transforms

LowLightSample = Tuple[Tensor, Tensor]


class LowLightDataset(Dataset[LowLightSample]):
    def __init__(self, path: str | Path, image_size: int) -> None:
        super().__init__()
        self.path: Path = Path(path)
        self.image_size: int = image_size
        self.transform: transforms.Compose = transforms.Compose(
            transforms=[
                transforms.Resize(size=(self.image_size, self.image_size)),
                transforms.ToTensor(),
            ]
        )

        self.low_path: Path = self.path / "low"
        self.high_path: Path = self.path / "high"

        self.low_datas: list[Path] = sorted(self.low_path.rglob(pattern="*.*"))
        self.high_datas: list[Path] = sorted(self.high_path.rglob(pattern="*.*"))

    def __len__(self) -> int:
        return len(self.low_datas)

    def __getitem__(self, index: int) -> LowLightSample:
        low_data: Path = self.low_datas[index]
        high_data: Path = self.high_path / low_data.name

        low_image: Image.Image = Image.open(fp=low_data).convert(mode="RGB")
        high_image: Image.Image = Image.open(fp=high_data).convert(mode="RGB")

        low_image, high_image = self._pair_augment(
            low_image=low_image, high_image=high_image
        )

        low_tensor: Tensor = cast(Tensor, self.transform(img=low_image))
        high_tensor: Tensor = cast(Tensor, self.transform(img=high_image))

        return low_tensor, high_tensor

    def _pair_augment(
        self, low_image: Image.Image, high_image: Image.Image
    ) -> tuple[Image.Image, Image.Image]:
        height, width = low_image.size
        min_crop_size = self.image_size

        if width >= min_crop_size and height >= min_crop_size:
            max_crop_size = min(width, height)
            new_crop_size = random.randint(min_crop_size, max_crop_size)

            left = random.randint(0, width - new_crop_size)
            top = random.randint(0, height - new_crop_size)
            right = left + new_crop_size
            bottom = top + new_crop_size

            low_image = low_image.crop((left, top, right, bottom))
            high_image = high_image.crop((left, top, right, bottom))

        if random.random() < 0.5:
            low_image = low_image.transpose(Image.FLIP_LEFT_RIGHT)
            high_image = high_image.transpose(Image.FLIP_LEFT_RIGHT)

        if random.random() < 0.5:
            low_image = low_image.transpose(Image.FLIP_TOP_BOTTOM)
            high_image = high_image.transpose(Image.FLIP_TOP_BOTTOM)

        return low_image, high_image

## data.dataloader

In [ ]:
from pathlib import Path
from typing import Literal, overload

import lightning as L
from torch.utils.data import ConcatDataset, DataLoader

LowLightDataLoader = DataLoader[LowLightSample]


class LowLightDataModule(L.LightningDataModule):
    def __init__(
        self,
        train_dir: str,
        valid_dir: str,
        bench_dir: str,
        infer_dir: str,
        image_size: int,
        batch_size: int = 32,
        num_workers: int = 4,
    ) -> None:
        super().__init__()
        self.train_dir: Path = Path(train_dir)
        self.valid_dir: Path = Path(valid_dir)
        self.bench_dir: Path = Path(bench_dir)
        self.infer_dir: Path = Path(infer_dir)

        self.image_size: int = image_size
        self.batch_size: int = batch_size
        self.num_workers: int = num_workers

        self.train_datasets: list[LowLightDataset] = []
        self.valid_datasets: list[LowLightDataset] = []
        self.bench_datasets: list[LowLightDataset] = []
        self.infer_datasets: list[LowLightDataset] = []

    def setup(
        self,
        stage: str | None = None,
    ) -> None:
        if stage is None:
            self.train_datasets = self._set_dataset(data_dir=self.train_dir)
            self.valid_datasets = self._set_dataset(data_dir=self.valid_dir)
            self.bench_datasets = self._set_dataset(data_dir=self.bench_dir)
            self.infer_datasets = self._set_dataset(data_dir=self.infer_dir)
        elif stage == "fit":
            self.train_datasets = self._set_dataset(data_dir=self.train_dir)
            self.valid_datasets = self._set_dataset(data_dir=self.valid_dir)
        elif stage == "validate":
            self.valid_datasets = self._set_dataset(data_dir=self.valid_dir)
        elif stage == "test":
            self.bench_datasets = self._set_dataset(data_dir=self.bench_dir)
        elif stage == "predict":
            self.infer_datasets = self._set_dataset(data_dir=self.infer_dir)
        else:
            raise ValueError(f"Invalid stage: {stage}")

    def _set_dataset(
        self,
        data_dir: Path,
    ) -> list[LowLightDataset]:
        datasets: list[LowLightDataset] = []
        for folder in data_dir.iterdir():
            if folder.is_dir():
                datasets.append(
                    LowLightDataset(
                        path=folder,
                        image_size=self.image_size,
                    )
                )
        return datasets

    @overload
    def _set_dataloader(
        self,
        datasets: list[LowLightDataset],
        concat: Literal[True],
        shuffle: bool = False,
    ) -> LowLightDataLoader: ...

    @overload
    def _set_dataloader(
        self,
        datasets: list[LowLightDataset],
        concat: Literal[False] = False,
        shuffle: bool = False,
    ) -> list[LowLightDataLoader]: ...

    def _set_dataloader(
        self,
        datasets: list[LowLightDataset],
        concat: bool = False,
        shuffle: bool = False,
    ) -> LowLightDataLoader | list[LowLightDataLoader]:
        if concat:
            dataset_concat: ConcatDataset[LowLightSample] = ConcatDataset(
                datasets=datasets,
            )
            dataloader: LowLightDataLoader = DataLoader(
                dataset=dataset_concat,
                batch_size=self.batch_size,
                shuffle=shuffle,
                num_workers=self.num_workers,
                persistent_workers=self.num_workers > 0,
                pin_memory=True,
            )
            return dataloader
        dataloaders: list[LowLightDataLoader] = []
        for dataset in datasets:
            loader: LowLightDataLoader = DataLoader(
                dataset=dataset,
                batch_size=self.batch_size,
                shuffle=shuffle,
                num_workers=self.num_workers,
                persistent_workers=self.num_workers > 0,
                pin_memory=True,
            )
            dataloaders.append(loader)
        return dataloaders

    def train_dataloader(self) -> LowLightDataLoader:
        return self._set_dataloader(
            datasets=self.train_datasets,
            concat=True,
            shuffle=True,
        )

    def val_dataloader(self) -> LowLightDataLoader:
        return self._set_dataloader(
            datasets=self.valid_datasets,
            concat=True,
        )

    def test_dataloader(self) -> list[LowLightDataLoader]:
        return self._set_dataloader(
            datasets=self.bench_datasets,
        )

    def predict_dataloader(self) -> list[LowLightDataLoader]:
        return self._set_dataloader(
            datasets=self.infer_datasets,
        )

In [ ]:
from abc import ABC, abstractmethod
from pathlib import Path
from typing import Any

from lightning import LightningDataModule, LightningModule, Trainer
from torch import Tensor


class _BaseRunner(ABC):
    def __init__(
        self,
        model: LightningModule,
        trainer: Trainer,
        hparams: dict[str, Any],
    ) -> None:
        self.trainer: Trainer = trainer
        self.hparams: dict[str, Any] = hparams
        self.log_dir: str = self.hparams.get("log_dir", "runs/")
        self.experiment_name: str = self.hparams.get("experiment_name", "test/")
        self.inference: str = self.hparams.get("inference", "inference/")
        self.out_dir: Path = Path(self.log_dir) / self.experiment_name / self.inference

        self.model: LightningModule = model
        self.datamodule: LightningDataModule = self._build_datamodule()

    def _build_datamodule(self) -> LowLightDataModule:
        datamodule: LowLightDataModule = LowLightDataModule(
            train_dir=self.hparams.get("train_data_path", "data/1_train"),
            valid_dir=self.hparams.get("valid_data_path", "data/2_valid"),
            bench_dir=str(self.hparams.get("bench_data_path", "data/3_bench")),
            infer_dir=str(self.hparams.get("infer_data_path", "data/4_infer")),
            image_size=self.hparams.get("image_size", 256),
            batch_size=self.hparams.get("batch_size", 16),
            num_workers=self.hparams.get("num_workers", 10),
        )

        return datamodule

    @abstractmethod
    def run(self) -> None:
        raise NotImplementedError


class LightningTrainer(_BaseRunner):
    def run(self) -> None:
        print("[INFO] Start Training...")
        self.trainer.fit(
            model=self.model,
            datamodule=self.datamodule,
        )
        print("[INFO] Training Completed.")


class LightningValidater(_BaseRunner):
    def run(self) -> None:
        print("[INFO] Start Validating...")
        self.trainer.validate(
            model=self.model,
            datamodule=self.datamodule,
        )
        print("[INFO] Validation Completed.")


class LightningBenchmarker(_BaseRunner):
    def run(self) -> None:
        print("[INFO] Start Benchmarking...")
        self.trainer.test(
            model=self.model,
            datamodule=self.datamodule,
        )
        print("[INFO] Benchmark Completed.")


class LightningInferencer(_BaseRunner):
    def run(self) -> None:
        print("[INFO] Start Inferencing...")
        output_batches: list[list[Tensor]] = cast(
            list[list[Tensor]],
            self.trainer.predict(
                model=self.model,
                datamodule=self.datamodule,
            ),
        )
        save_images(batch_list=output_batches, out_dir=self.out_dir)
        print("[INFO] Inference Completed.")

In [ ]:
from typing import Any

from lightning import LightningModule, Trainer, seed_everything
from lightning.pytorch.callbacks import (
    Callback,
    EarlyStopping,
    LearningRateMonitor,
    ModelCheckpoint,
)
from lightning.pytorch.loggers import TensorBoardLogger


class LightningEngine:
    def __init__(
        self,
        model_class: type[LightningModule],
        hparams: dict[str, Any],
        checkpoint_path: str | None = None,
    ) -> None:
        self.hparams: dict[str, Any] = hparams
        self.checkpoint_path: str | None = checkpoint_path

        seed_everything(seed=self.hparams.get("seed", 42), workers=True)

        if checkpoint_path:
            self.model = model_class.load_from_checkpoint(
                checkpoint_path=checkpoint_path,
            )
        else:
            self.model = model_class(hparams=self.hparams)

        self.logger: TensorBoardLogger = self._build_logger()
        self.callbacks: list[Callback] = self._build_callbacks()
        self.trainer: Trainer = self._build_trainer()

    def _build_trainer(self) -> Trainer:
        return Trainer(
            max_epochs=self.hparams.get("max_epochs", 100),
            accelerator=self.hparams.get("accelerator", "gpu"),
            devices=self.hparams.get("devices", 1),
            precision=self.hparams.get("precision", "16-mixed"),
            log_every_n_steps=self.hparams.get("log_every_n_steps", 5),
            logger=self.logger,
            callbacks=self.callbacks,
            benchmark=False,
            deterministic=True,
        )

    def _build_logger(self) -> TensorBoardLogger:
        return TensorBoardLogger(
            save_dir=self.hparams.get("log_dir", "runs/"),
            name=self.hparams.get("experiment_name", "test/"),
        )

    def _build_callbacks(self) -> list[Callback]:
        callbacks: list[Callback] = [
            ModelCheckpoint(
                monitor="valid/loss_total",
                save_top_k=1,
                mode="min",
                filename="best",
            ),
            ModelCheckpoint(
                every_n_epochs=5,
                save_top_k=-1,
                filename="epoch-{epoch:02d}",
            ),
            EarlyStopping(
                monitor="valid/loss_total",
                patience=self.hparams.get("patience", 25),
                mode="min",
                verbose=True,
            ),
            LearningRateMonitor(logging_interval="step"),
        ]
        return callbacks

    def _create_and_run_runner(
        self,
        runner_class: type[_BaseRunner],
    ) -> None:
        runner: _BaseRunner = runner_class(
            model=self.model,
            trainer=self.trainer,
            hparams=self.hparams,
        )
        runner.run()

    def train(self) -> None:
        self._create_and_run_runner(runner_class=LightningTrainer)

    def valid(self) -> None:
        self._create_and_run_runner(runner_class=LightningValidater)

    def bench(self) -> None:
        self._create_and_run_runner(runner_class=LightningBenchmarker)

    def infer(self) -> None:
        self._create_and_run_runner(runner_class=LightningInferencer)

# model

## model.block

### model.block.homomorphic

In [ ]:
import math
from typing import Tuple

import torch
import torch.nn as nn
from torch import Tensor


def safe_tensor(tensor: Tensor, name: str = "tensor") -> Tensor:
    """텐서에 NaN 또는 Inf가 있는지 확인하고, 있으면 0.0으로 변환합니다."""
    if not torch.isfinite(tensor).all():
        print(
            f"--- [WARNING] '{name}' 텐서에서 NaN/Inf가 감지되었습니다! 0.0으로 강제 변환합니다. ---"
        )
        return torch.nan_to_num(tensor, nan=0.0, posinf=0.0, neginf=0.0)
    return tensor


class RGB2YCrCbBlock(nn.Module):
    def __init__(
        self,
        offset: float,
    ) -> None:
        super().__init__()
        self.conv = nn.Conv2d(in_channels=3, out_channels=3, kernel_size=1, bias=True)
        transform = torch.tensor(
            data=[
                [0.299, 0.587, 0.114],
                [0.5, -0.418688, -0.081312],
                [-0.168736, -0.331264, 0.5],
            ],
            dtype=torch.float32,
        ).view(3, 3, 1, 1)
        bias = torch.tensor(data=[0.0, offset, offset], dtype=torch.float32)
        self.conv.weight = nn.Parameter(transform, requires_grad=False)
        self.conv.bias = nn.Parameter(bias, requires_grad=False)

    def forward(self, x: Tensor) -> Tuple[Tensor, Tensor, Tensor]:
        x = safe_tensor(x, name="RGB2YCrCb.input_x")

        ycrcb: Tensor = self.conv(x)

        ycrcb = safe_tensor(ycrcb, name="RGB2YCrCb.output_ycrcb")

        y, cr, cb = torch.chunk(input=ycrcb, chunks=3, dim=1)
        return y, cr, cb


class YCrCb2RGBBlock(nn.Module):
    def __init__(
        self,
        offset: float,
    ) -> None:
        super().__init__()
        self.conv = nn.Conv2d(in_channels=3, out_channels=3, kernel_size=1, bias=False)
        transform = torch.tensor(
            data=[
                [1.0, 1.403, 0.0],
                [1.0, -0.714, -0.344],
                [1.0, 0.0, 1.773],
            ],
            dtype=torch.float32,
        ).view(3, 3, 1, 1)
        self.conv.weight = nn.Parameter(transform, requires_grad=False)
        bias = torch.tensor(data=[0.0, offset, offset], dtype=torch.float32).view(
            1, 3, 1, 1
        )
        self.register_buffer(name="chrominance_bias", tensor=bias)

    def forward(
        self,
        y: Tensor,
        cr: Tensor,
        cb: Tensor,
    ) -> Tensor:
        y = safe_tensor(y, name="YCrCb2RGB.input_y")
        cr = safe_tensor(cr, name="YCrCb2RGB.input_cr")
        cb = safe_tensor(cb, name="YCrCb2RGB.input_cb")

        ycrcb: Tensor = torch.cat(tensors=[y, cr, cb], dim=1)

        bias = self.chrominance_bias.to(dtype=ycrcb.dtype, device=ycrcb.device)
        centered: Tensor = ycrcb - bias

        rgb: Tensor = self.conv(centered)

        rgb = safe_tensor(rgb, name="YCrCb2RGB.output_rgb")

        return rgb


def get_gaussian_kernel(kernel_size=5, sigma=1, channels=1):
    x_cord = torch.arange(kernel_size)
    x_grid = x_cord.repeat(kernel_size).view(kernel_size, kernel_size)
    y_grid = x_grid.t()
    xy_grid = torch.stack([x_grid, y_grid], dim=-1).float()
    mean = (kernel_size - 1) / 2.0
    variance = sigma**2.0
    gaussian_kernel = (1.0 / (2.0 * math.pi * variance)) * torch.exp(
        -torch.sum((xy_grid - mean) ** 2.0, dim=-1) / (2 * variance)
    )
    gaussian_kernel = gaussian_kernel / torch.sum(gaussian_kernel)
    gaussian_kernel = gaussian_kernel.view(1, 1, kernel_size, kernel_size)
    gaussian_kernel = gaussian_kernel.repeat(channels, 1, 1, 1)
    return gaussian_kernel


class HomomorphicSeparationBlock(nn.Module):
    def __init__(
        self,
        cutoff: float,
        kernel_size: int = 21,
        sigma: float = 5.0,
    ) -> None:
        super().__init__()
        gaussian_weights = get_gaussian_kernel(
            kernel_size=kernel_size, sigma=sigma, channels=1
        )
        self.gaussian_blur = nn.Conv2d(
            in_channels=1,
            out_channels=1,
            kernel_size=kernel_size,
            padding=(kernel_size // 2),
            bias=False,
        )
        self.gaussian_blur.weight = nn.Parameter(gaussian_weights, requires_grad=False)

    def forward(
        self,
        x: Tensor,
    ) -> Tuple[Tensor, Tensor]:
        original_dtype = x.dtype

        x = safe_tensor(x, name="HSB.input_x")

        x_f32: Tensor = x.float()
        x_clamped_f32: Tensor = torch.clamp(x_f32, min=1e-5)
        x_log_f32: Tensor = torch.log(x_clamped_f32)
        x_log_f32 = torch.nan_to_num(x_log_f32, nan=0.0)

        x_log_original: Tensor = x_log_f32.to(original_dtype)
        low_log_original: Tensor = self.gaussian_blur(x_log_original)

        low_log_original = safe_tensor(low_log_original, name="HSB.conv_output")

        low_log_f32: Tensor = low_log_original.float()
        LOG_CLAMP_MAX = 80.0
        low_log_f32 = torch.clamp(low_log_f32, max=LOG_CLAMP_MAX)
        il_f32: Tensor = torch.exp(low_log_f32)

        il_f32_safe = torch.clamp(il_f32, min=1e-6)
        re_f32: Tensor = x_clamped_f32 / il_f32_safe

        re_f32 = torch.nan_to_num(re_f32, nan=0.0, posinf=1.0)
        re_f32 = torch.clamp(re_f32, 0.0, 1.0)

        return il_f32.to(original_dtype), re_f32.to(original_dtype)


class ImageDecomposition(nn.Module):
    def __init__(
        self,
        offset: float,
        cutoff: float,
    ) -> None:
        super().__init__()
        self.rgb2ycrcb: RGB2YCrCbBlock = RGB2YCrCbBlock(
            offset=offset,
        )
        self.homomorphic: HomomorphicSeparationBlock = HomomorphicSeparationBlock(
            cutoff=cutoff,
        )

    def forward(
        self,
        x: Tensor,
    ) -> Tuple[Tensor, Tensor, Tensor, Tensor, Tensor]:
        x = safe_tensor(x, name="Decomposition.input_x")

        y, cr, cb = self.rgb2ycrcb(x)

        y = safe_tensor(y, name="Decomposition.output_y")
        cr = safe_tensor(cr, name="Decomposition.output_cr")
        cb = safe_tensor(cb, name="Decomposition.output_cb")

        il, re = self.homomorphic(y)

        il = safe_tensor(il, name="Decomposition.output_il")
        re = safe_tensor(re, name="Decomposition.output_re")

        return y, cr, cb, il, re


class ImageComposition(nn.Module):
    def __init__(
        self,
        offset: float,
    ) -> None:
        super().__init__()
        self.ycrcb2rgb: YCrCb2RGBBlock = YCrCb2RGBBlock(
            offset=offset,
        )

    def forward(
        self,
        cr: Tensor,
        cb: Tensor,
        il: Tensor,
        re: Tensor,
    ) -> Tuple[Tensor, Tensor]:
        cr = safe_tensor(cr, name="Composition.input_cr")
        cb = safe_tensor(cb, name="Composition.input_cb")
        il = safe_tensor(il, name="Composition.input_il (network output)")
        re = safe_tensor(re, name="Composition.input_re")

        y_enh: Tensor = il * re
        y_enh = safe_tensor(y_enh, name="Composition.y_enh (il * re)")

        img_enh: Tensor = self.ycrcb2rgb(y_enh, cr, cb)

        img_enh = safe_tensor(img_enh, name="Composition.output_img_enh")

        return img_enh, y_enh

### model.block.illuminationenhancer

In [ ]:
import torch.nn as nn
from torch import Tensor


class ResidualBlock(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
    ) -> None:
        super().__init__()
        self.conv1: nn.Conv2d = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.bn1: nn.BatchNorm2d = nn.BatchNorm2d(num_features=out_channels)
        self.act1: nn.ReLU = nn.ReLU()

        self.conv2: nn.Conv2d = nn.Conv2d(
            in_channels=out_channels,
            out_channels=out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.bn2: nn.BatchNorm2d = nn.BatchNorm2d(num_features=out_channels)
        self.act2: nn.ReLU = nn.ReLU()

        if in_channels != out_channels:
            self.skip_proj: nn.Module = nn.Conv2d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=1,
            )
        else:
            self.skip_proj = nn.Identity()

    def forward(
        self,
        x: Tensor,
    ) -> Tensor:
        x1: Tensor = self.act1(self.bn1(self.conv1(x)))
        x2: Tensor = self.act2(self.bn2(self.conv2(x1)))

        residual: Tensor = self.skip_proj(x) + x2
        return residual


class DoubleConv(nn.Module):
    def __init__(
        self,
        in_channels: int,
        hidden_channels: int,
        out_channels: int,
    ) -> None:
        super().__init__()
        self.conv1: ResidualBlock = ResidualBlock(
            in_channels=in_channels,
            out_channels=hidden_channels,
        )
        self.conv2: ResidualBlock = ResidualBlock(
            in_channels=hidden_channels,
            out_channels=out_channels,
        )

    def forward(
        self,
        x: Tensor,
    ) -> Tensor:
        b, c, h, w = x.shape
        x = self.conv1(x)
        x = self.conv2(x)
        return x


class Downsampling(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
    ) -> None:
        super().__init__()
        self.conv: nn.Conv2d = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=2,
            stride=2,
            padding=0,
            bias=False,
        )

    def forward(
        self,
        x: Tensor,
    ) -> Tensor:
        return self.conv(x)


class Upsampling(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
    ) -> None:
        super().__init__()
        self.conv: nn.ConvTranspose2d = nn.ConvTranspose2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=2,
            stride=2,
            padding=0,
            bias=False,
        )

    def forward(
        self,
        x: Tensor,
    ) -> Tensor:
        return self.conv(x)


class IlluminationEnhancer(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        hidden_channels: int,
        num_resolution: int,
    ) -> None:
        super().__init__()

        self.in_conv: nn.Conv2d = nn.Conv2d(
            in_channels=in_channels,
            out_channels=hidden_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        hidden_level = hidden_channels
        down: list[nn.Module] = []
        for level in range(num_resolution):
            down.append(
                DoubleConv(
                    in_channels=hidden_level,
                    hidden_channels=hidden_level,
                    out_channels=hidden_level,
                )
            )
            down.append(
                Downsampling(
                    in_channels=hidden_level,
                    out_channels=hidden_level * 2,
                )
            )
            hidden_level *= 2
        self.down: nn.ModuleList = nn.ModuleList(modules=down)

        mid: list[nn.Module] = []
        for _ in range(num_resolution // 2):
            mid.append(
                DoubleConv(
                    in_channels=hidden_level,
                    hidden_channels=hidden_level,
                    out_channels=hidden_level,
                )
            )
        self.mid: nn.ModuleList = nn.ModuleList(modules=mid)

        up: list[nn.Module] = []
        for level in range(num_resolution):
            up.append(
                Upsampling(
                    in_channels=hidden_level,
                    out_channels=hidden_level // 2,
                )
            )
            up.append(
                DoubleConv(
                    in_channels=hidden_level,
                    hidden_channels=hidden_level,
                    out_channels=hidden_level // 2,
                )
            )
            hidden_level //= 2

        self.up: nn.ModuleList = nn.ModuleList(modules=up)

        self.out_conv: nn.Conv2d = nn.Conv2d(
            in_channels=hidden_channels,
            out_channels=out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )

    def forward(
        self,
        x: Tensor,
    ) -> Tensor:
        x = self.in_conv(x)

        residuals: list[Tensor] = []
        for module in self.down:
            if isinstance(module, Downsampling):
                residuals.append(x)
                x = module(x)
            else:
                x = module(x)

        for module in self.mid:
            x = module(x)

        for module in self.up:
            if isinstance(module, Upsampling):
                x = module(x)
                x = torch.cat(tensors=[x, residuals.pop()], dim=1)
            else:
                x = module(x)

        x = self.out_conv(x)
        return x

### model.block.lowlightenhancer

In [ ]:
import torch.nn as nn
from torch import Tensor


class LowLightEnhancer(nn.Module):
    def __init__(
        self,
        hidden_channels: int,
        num_resolution: int,
        cutoff: float,
        offset: float,
    ) -> None:
        super().__init__()

        self.decomposition: ImageDecomposition = ImageDecomposition(
            offset=offset,
            cutoff=cutoff,
        )

        self.illumination_enhancer: IlluminationEnhancer = IlluminationEnhancer(
            in_channels=1,
            out_channels=1,
            hidden_channels=hidden_channels,
            num_resolution=num_resolution,
        )

        self.composition: ImageComposition = ImageComposition(
            offset=offset,
        )

    def forward(self, low: Tensor) -> dict[str, Tensor]:
        y, cr, cb, il, re = self.decomposition(low)

        il_enh = self.illumination_enhancer(il)

        img_enh, y_enh = self.composition(
            cr,
            cb,
            il_enh,
            re,
        )
        img_enh = torch.clamp(input=img_enh, min=0.0, max=1.0)

        outputs: dict[str, Tensor] = {
            "low_luminance": y,
            "low_chroma_red": cr,
            "low_chroma_blue": cb,
            "low_illuminance": il,
            "low_reflectance": re,
            "low_rgb": low,
            "enh_illuminance": il_enh,
            "enh_luminance": y_enh,
            "enh_rgb": img_enh,
        }
        return outputs

## model.loss

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor


class ColorConstancyLoss(nn.Module):
    def __init__(self) -> None:
        super().__init__()

    def forward(
        self,
        input: Tensor,
    ) -> Tensor:
        if input.shape[1] == 1:
            return torch.tensor(data=0.0, device=input.device, dtype=input.dtype)

        mean_rgb = torch.mean(input=input, dim=[2, 3], keepdim=True)
        mr, mg, mb = torch.split(tensor=mean_rgb, split_size_or_sections=1, dim=1)
        Drg = (mr - mg) ** 2
        Drb = (mr - mb) ** 2
        Dgb = (mb - mg) ** 2

        loss = (Drg + Drb + Dgb).mean()
        return loss


class SpatialConsistencyLoss(nn.Module):
    def __init__(self) -> None:
        super().__init__()

        kernel_l = (
            torch.tensor(
                data=[[0, 0, 0], [-1, 1, 0], [0, 0, 0]],
                dtype=torch.float32,
            )
            .unsqueeze(dim=0)
            .unsqueeze(dim=0)
        )
        kernel_r = (
            torch.tensor(
                data=[[0, 0, 0], [0, 1, -1], [0, 0, 0]],
                dtype=torch.float32,
            )
            .unsqueeze(dim=0)
            .unsqueeze(dim=0)
        )
        kernel_u = (
            torch.tensor(
                data=[[0, -1, 0], [0, 1, 0], [0, 0, 0]],
                dtype=torch.float32,
            )
            .unsqueeze(dim=0)
            .unsqueeze(dim=0)
        )
        kernel_d = (
            torch.tensor(
                data=[[0, 0, 0], [0, 1, 0], [0, -1, 0]],
                dtype=torch.float32,
            )
            .unsqueeze(dim=0)
            .unsqueeze(dim=0)
        )

        self.register_buffer(name="weight_l", tensor=kernel_l)
        self.register_buffer(name="weight_r", tensor=kernel_r)
        self.register_buffer(name="weight_u", tensor=kernel_u)
        self.register_buffer(name="weight_d", tensor=kernel_d)

        self.pool = nn.AvgPool2d(kernel_size=4)

    def forward(
        self,
        input: Tensor,
        target: Tensor,
    ) -> Tensor:
        if input.shape[1] > 1:
            org_mean = torch.mean(input=input, dim=1, keepdim=True)
            enh_mean = torch.mean(input=target, dim=1, keepdim=True)

        else:
            org_mean = input
            enh_mean = target

        org_pool = self.pool(org_mean)
        enh_pool = self.pool(enh_mean)

        D_org_l = F.conv2d(input=org_pool, weight=self.weight_l, padding=1)
        D_org_r = F.conv2d(input=org_pool, weight=self.weight_r, padding=1)
        D_org_u = F.conv2d(input=org_pool, weight=self.weight_u, padding=1)
        D_org_d = F.conv2d(input=org_pool, weight=self.weight_d, padding=1)

        D_enh_l = F.conv2d(input=enh_pool, weight=self.weight_l, padding=1)
        D_enh_r = F.conv2d(input=enh_pool, weight=self.weight_r, padding=1)
        D_enh_u = F.conv2d(input=enh_pool, weight=self.weight_u, padding=1)
        D_enh_d = F.conv2d(input=enh_pool, weight=self.weight_d, padding=1)

        loss = (
            (D_org_l - D_enh_l) ** 2
            + (D_org_r - D_enh_r) ** 2
            + (D_org_u - D_enh_u) ** 2
            + (D_org_d - D_enh_d) ** 2
        ).mean()

        return loss


class ExposureLoss(nn.Module):
    def __init__(
        self,
        patch_size: int = 16,
        mean_val: float = 0.8,
    ) -> None:
        super().__init__()
        self.pool = nn.AvgPool2d(kernel_size=patch_size)
        self.register_buffer(
            name="mean_val_tensor",
            tensor=torch.tensor(
                data=[mean_val],
                dtype=torch.float32,
            ),
        )

    def forward(
        self,
        input: Tensor,
    ) -> Tensor:
        if input.shape[1] > 1:
            x = torch.mean(input=input, dim=1, keepdim=True)

        else:
            x = input

        x_pool = self.pool(x)
        loss = torch.mean(input=(x_pool - self.mean_val_tensor) ** 2)
        return loss


class IlluminationSmoothnessLoss(nn.Module):
    def __init__(
        self,
        weight: float = 1e-3,
    ) -> None:
        super().__init__()
        self.weight = weight

    def forward(
        self,
        input: Tensor,
    ) -> Tensor:
        batch = input.size(dim=0)
        h = input.size(dim=2)
        w = input.size(dim=3)

        if h == 1 or w == 1:
            return torch.tensor(data=0.0, device=input.device, dtype=input.dtype)

        count_h = (h - 1) * w
        count_w = h * (w - 1)

        h_tv = ((input[:, :, 1:, :] - input[:, :, : h - 1, :]) ** 2).sum()
        w_tv = ((input[:, :, :, 1:] - input[:, :, :, : w - 1]) ** 2).sum()

        loss = self.weight * 2 * (h_tv / count_h + w_tv / count_w) / batch

        return loss


class TotalLoss(nn.Module):
    def __init__(
        self,
        lambda_spa: float = 1.0,
        lambda_exp: float = 1.0,
        lambda_col: float = 1.0,
        lambda_illum: float = 1.0,
        exp_patch_size: int = 16,
        exp_mean_val: float = 0.8,
        illum_weight: float = 1.0,
    ) -> None:
        super().__init__()

        self.lambda_spa = lambda_spa
        self.lambda_exp = lambda_exp
        self.lambda_col = lambda_col
        self.lambda_illum = lambda_illum

        self.loss_spa = SpatialConsistencyLoss()
        self.loss_exp = ExposureLoss(patch_size=exp_patch_size, mean_val=exp_mean_val)
        self.loss_col = ColorConstancyLoss()
        self.loss_illum = IlluminationSmoothnessLoss(weight=illum_weight)

    def forward(
        self,
        low_luminance: Tensor,
        enh_luminance: Tensor,
        enh_illuminance: Tensor,
        enh_rgb: Tensor,
    ) -> tuple[Tensor, dict[str, Tensor]]:
        l_spa = self.loss_spa(low_luminance, enh_luminance)

        l_exp = self.loss_exp(enh_illuminance)

        l_illum = self.loss_illum(enh_illuminance)

        l_col = self.loss_col(enh_rgb)

        total_loss = (
            self.lambda_spa * l_spa
            + self.lambda_exp * l_exp
            + self.lambda_col * l_col
            + self.lambda_illum * l_illum
        )

        loss_dict = {
            "loss_total": total_loss.detach(),
            "loss_spa": self.lambda_spa * l_spa.detach(),
            "loss_exp": self.lambda_exp * l_exp.detach(),
            "loss_col": self.lambda_col * l_col.detach(),
            "loss_illum": self.lambda_illum * l_illum.detach(),
        }

        return total_loss, loss_dict

## model.model

In [ ]:
from typing import Any, Literal

import lightning as L
from torch import Tensor
from torch.optim.adam import Adam
from torch.optim.optimizer import Optimizer


class LowLightEnhancerLightning(L.LightningModule):
    def __init__(self, hparams: dict[str, Any]) -> None:
        super().__init__()
        torch.autograd.set_detect_anomaly(mode=True)
        self.save_hyperparameters(hparams)

        self.model: LowLightEnhancer = LowLightEnhancer(
            hidden_channels=self.hparams.get("hidden_channels", 32),
            num_resolution=self.hparams.get("num_resolution", 4),
            offset=self.hparams.get("offset", 0.5),
            cutoff=self.hparams.get("cutoff", 0.1),
        )

        self.loss: TotalLoss = TotalLoss(
            lambda_spa=self.hparams.get("lambda_spa", 10.0),
            lambda_exp=self.hparams.get("lambda_exp", 100.0),
            lambda_col=self.hparams.get("lambda_col", 1000.0),
            lambda_illum=self.hparams.get("lambda_illum", 1000.0),
            exp_patch_size=self.hparams.get("exp_patch_size", 4),
            exp_mean_val=self.hparams.get("exp_mean_val", 0.6),
            illum_weight=self.hparams.get("illum_weight", 1e-3),
        ).eval()

        self.metric = ImageQualityMetrics().eval()

    def forward(self, low: Tensor) -> dict[str, Tensor]:
        return self.model(low)

    def _calculate_loss(
        self,
        outputs: dict[str, Tensor],
    ) -> tuple[Tensor, dict[str, Tensor]]:
        low_lum = outputs["low_luminance"]
        enh_lum = outputs["enh_luminance"]
        enh_illum = outputs["enh_illuminance"]
        enh_rgb_out = outputs["enh_rgb"]

        total_loss, loss_dict = self.loss(
            low_luminance=low_lum,
            enh_luminance=enh_lum,
            enh_illuminance=enh_illum,
            enh_rgb=enh_rgb_out,
        )
        return total_loss, loss_dict

    def _shared_step(
        self,
        batch: LowLightSample,
    ) -> tuple[dict[str, Tensor], Tensor, dict[str, Tensor]]:
        low_img, _ = batch
        outputs = self.forward(low=low_img)
        total_loss, loss_dict = self._calculate_loss(outputs=outputs)
        return outputs, total_loss, loss_dict

    def _logging(
        self,
        stage: Literal["train", "valid"],
        outputs: dict[str, Tensor],
        loss_dict: dict[str, Tensor],
        batch_idx: int,
    ) -> None:
        if batch_idx % 50 != 0:
            return

        logger = self.logger.experiment

        image_keys = [
            "low_luminance",
            "low_chroma_red",
            "low_chroma_blue",
            "low_illuminance",
            "low_reflectance",
            "low_rgb",
            "enh_illuminance",
            "enh_luminance",
            "enh_rgb",
        ]

        for i, key in enumerate(iterable=image_keys):
            if key in outputs:
                logger.add_images(
                    f"{stage}/{i + 1}_{key}", outputs[key], self.global_step
                )

        log_dict = {f"{stage}/{k}": v for k, v in loss_dict.items()}
        self.log_dict(dictionary=log_dict, prog_bar=True)

    def training_step(
        self,
        batch: LowLightSample,
        batch_idx: int,
    ) -> Tensor:
        outputs, total_loss, loss_dict = self._shared_step(batch=batch)
        self._logging(
            stage="train", outputs=outputs, loss_dict=loss_dict, batch_idx=batch_idx
        )
        return total_loss

    def validation_step(
        self,
        batch: LowLightSample,
        batch_idx: int,
    ) -> Tensor:
        outputs, total_loss, loss_dict = self._shared_step(batch=batch)
        self._logging(
            stage="valid", outputs=outputs, loss_dict=loss_dict, batch_idx=batch_idx
        )
        return total_loss

    def test_step(
        self,
        batch: LowLightSample,
        batch_idx: int,
        dataloader_idx: int = 0,
    ) -> None:
        low_img, high_img = batch
        outputs = self.forward(low=low_img)

        metrics = self.metric.full(preds=outputs["enh_rgb"], targets=high_img)

        self.log_dict(
            dictionary={
                "test/PSNR": metrics["PSNR"],
                "test/SSIM": metrics["SSIM"],
                "test/LPIPS": metrics["LPIPS"],
                "test/NIQE": metrics["NIQE"],
                "test/BRISQUE": metrics["BRISQUE"],
            },
            prog_bar=True,
        )

    def predict_step(
        self,
        batch: LowLightSample,
        batch_idx: int,
        dataloader_idx: int = 0,
    ) -> list[Tensor]:
        low_img, _ = batch
        results = self.forward(low=low_img)
        return [results["enh_rgb"]]

    def configure_optimizers(self) -> list[Optimizer]:
        lr = float(self.hparams.get("lr", 5e-4))

        optimizer = Adam(
            params=self.parameters(),
            lr=lr,
            betas=self.hparams.get("betas", (0.9, 0.999)),
            eps=self.hparams.get("eps", 1e-8),
            weight_decay=self.hparams.get("weight_decay", 0.0),
        )
        return [optimizer]

# main

In [ ]:
from typing import Any

import numpy as np


def get_hparams() -> dict[str, Any]:
    hparams: dict[str, Any] = {
        # Engine
        "seed": 42,
        "max_epochs": 200,
        "accelerator": "gpu",
        "devices": 1,
        "precision": "16-mixed",
        # "precision": 32,
        "log_every_n_steps": 5,
        "log_dir": "runs/",
        "experiment_name": "test/",
        "patience": 100,
        # Runner
        "inference": "inference/",
        "train_data_path": "data/1_train",
        "valid_data_path": "data/2_valid",
        "bench_data_path": "data/3_bench",
        "infer_data_path": "data/4_infer",
        "image_size": 512,
        "batch_size": 16,
        "num_workers": 10,
        # Model
        "hidden_channels": 64,
        "num_resolution": 4,
        "offset": 0.5,
        "cutoff": 0.1,
    }
    return hparams


def main() -> None:
    hparams: dict[str, Any] = get_hparams()
    seed: int = random.randint(0, 1000)
    hparams["seed"] = seed

    for i in np.arange(0.05, 0.5, 0.05):
        hparams["cutoff"] = i
        engine: LightningEngine = LightningEngine(
            model_class=LowLightEnhancerLightning,
            hparams=hparams,
        )
        engine.train()
        engine.bench()


if __name__ == "__main__":
    main()